# 1. Setup

In [1]:
!pip install replay-rec pyspark==3.5.1 lightning torchvision --upgrade --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 53.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.1/368.1 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 152.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.9 MB/s eta 0:0

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from pathlib import Path

ROOT = '/content/drive/MyDrive/Projects/ml-group-project'
DATA = os.path.join(ROOT, 'data')
ML_32M_DATA = os.path.join(DATA, 'ml-32m')

PARQUETS = Path(os.path.join(DATA, 'parquets'))
PARQUETS.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PARQUETS / "train.parquet"
VAL_PATH = PARQUETS / "val.parquet"
PREDICT_PATH = PARQUETS / "test.parquet"

ENCODER_PATH = PARQUETS / "encoder"

In [3]:
import numpy as np

np.random.seed(42)

# 2. Dataset

- [MovieLens 32M dataset](https://grouplens.org/datasets/movielens/)
- Load dataframes

In [5]:
import pandas as pd

ratings = pd.read_csv(os.path.join(ML_32M_DATA, 'ratings.csv'))
ratings.head(3)

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976


In [6]:
# ratings["timestamp"] = ratings["timestamp"].astype("int64")
# ratings = ratings.sort_values(by="timestamp")
# ratings["timestamp"] = ratings.groupby("userId").cumcount()
# ratings.head(3)

import pandas as pd
from sklearn.preprocessing import LabelEncoder

ratings = ratings.sort_values(["userId", "timestamp"]).reset_index(drop=True)

user_enc = LabelEncoder()
item_enc = LabelEncoder()

ratings["user_idx"] = user_enc.fit_transform(ratings["userId"])
ratings["item_idx"] = item_enc.fit_transform(ratings["movieId"])

# last item = test, second last = val, rest = train
train_rows = []
val_rows = []
test_rows = []

for _, g in ratings.groupby("user_idx"):
    if len(g) < 3:
        continue
    train_rows.append(g.iloc[:-2])
    val_rows.append(g.iloc[[-2]])
    test_rows.append(g.iloc[[-1]])

train_df = pd.concat(train_rows).reset_index(drop=True)
val_df = pd.concat(val_rows).reset_index(drop=True)
test_df = pd.concat(test_rows).reset_index(drop=True)

# Save the dataframes before splitting
train_df.to_parquet(PARQUETS / "raw_train.parquet")
val_df.to_parquet(PARQUETS / "raw_val.parquet")
test_df.to_parquet(PARQUETS / "raw_test.parquet")

In [8]:
from replay.splitters import LastNSplitter

# Rename columns to match replay-rec's default expectations
# (user_id, item_id, timestamp)

encoded_interactions = train_df.rename(
    columns={"user_idx": "user_id", "item_idx": "item_id"}
)

# encoded_interactions = train_df.rename(
#     columns={'userId': 'user_id', 'movieId': 'item_id'}
# )

splitter = LastNSplitter(
    N=1,
    divide_column="user_id", # Use the renamed column
    query_column="user_id",  # Use the renamed column
    strategy="interactions",
    drop_cold_users=True,
    drop_cold_items=True
)

test_events, test_gt = splitter.split(encoded_interactions)
validation_events, validation_gt = splitter.split(test_events)
train_events = validation_events

test_gt.to_parquet(PARQUETS / "test_gt.parquet")

In [9]:
from replay.data.nn.utils import groupby_sequences

def bake_data(full_data):
    return groupby_sequences(events=full_data, groupby_col="user_id", sort_col="timestamp")

In [10]:
train_events['user_id'] = train_events['user_id'] + 1
train_events = bake_data(train_events)

validation_events['user_id'] = validation_events['user_id'] + 1
validation_events = bake_data(validation_events)

validation_gt['user_id'] = validation_gt['user_id'] + 1
validation_gt = bake_data(validation_gt)

test_events['user_id'] = test_events['user_id'] + 1
test_events = bake_data(test_events)

test_gt['user_id'] = test_gt['user_id'] + 1
test_gt = bake_data(test_gt)
test_gt.to_parquet(PARQUETS / "test_gt.parquet")

train_events.head()

,user_id,userId,movieId,rating,timestamp,item_id
0,1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[2966, 2997, 2890, 3078, 2882, 541, 838, 1136,...","[1.0, 4.0, 4.0, 2.0, 1.0, 5.0, 5.0, 1.0, 2.0, ...","[943226846, 943226846, 943226916, 943226986, 9...","[2874, 2905, 2798, 2985, 2790, 536, 820, 1108,..."
1,2,"[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[592, 296, 380, 153, 344, 349, 588, 231, 318, ...","[4.0, 1.0, 5.0, 3.0, 1.0, 3.0, 5.0, 2.0, 5.0, ...","[836423201, 836423202, 836423202, 836423237, 8...","[584, 292, 375, 151, 339, 344, 580, 228, 314, ..."
2,3,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","[2012, 466, 2268, 168, 1544, 4306, 1485, 2617,...","[3.0, 1.0, 4.0, 3.5, 4.0, 3.5, 4.0, 4.0, 3.5, ...","[1084484354, 1084484362, 1084484382, 108448438...","[1923, 461, 2177, 166, 1489, 4202, 1440, 2526,..."
3,4,"[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[1210, 1833, 2745, 1272, 1327, 2115, 2683, 269...","[3.0, 2.0, 3.0, 4.0, 3.0, 5.0, 3.0, 2.0, 2.0, ...","[960485234, 960485234, 960485234, 960485281, 9...","[1179, 1746, 2653, 1239, 1293, 2025, 2591, 260..."
4,5,"[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[592, 150, 590, 296, 380, 165, 344, 349, 588, ...","[4.0, 3.0, 3.0, 1.0, 5.0, 4.0, 3.0, 4.0, 3.0, ...","[840763913, 840763914, 840763914, 840763915, 8...","[584, 148, 582, 292, 375, 163, 339, 344, 580, ..."


In [11]:
def add_gt_to_events(events_df, gt_df):
    gt_to_join = gt_df[["user_id", "item_id"]].rename(columns={"item_id": "ground_truth"})

    events_df = events_df.merge(gt_to_join, on="user_id", how="inner")
    return events_df

In [12]:
validation_events = add_gt_to_events(validation_events, validation_gt)
test_events = add_gt_to_events(test_events, test_gt)

In [13]:
import pickle

train_events.to_parquet(TRAIN_PATH)
validation_events.to_parquet(VAL_PATH)
test_events.to_parquet(PREDICT_PATH)

# encoder.save(ENCODER_PATH)

ENCODER_PATH.mkdir(parents=True, exist_ok=True)

with open(ENCODER_PATH / "user_encoder.pkl", "wb") as f:
    pickle.dump(user_enc, f)

with open(ENCODER_PATH / "item_encoder.pkl", "wb") as f:
    pickle.dump(item_enc, f)

# 3. Model

In [4]:
MAX_SEQ_LEN = 50
NUM_BLOCKS = 2
NUM_HEADS = 2
DROPOUT = 0.3
BATCH_SIZE = 32
EMBEDDING_DIM = 64

In [5]:
import pickle

with open(ENCODER_PATH / "item_encoder.pkl", "rb") as f:
    item_enc = pickle.load(f)

In [6]:
from replay.data import FeatureHint, FeatureType
from replay.data.nn.schema import TensorFeatureInfo, TensorSchema

import torch
import lightning as L
from replay.nn.transform.template import make_default_sasrec_transforms

from replay.data.nn import ParquetModule

from replay.nn.sequential import SasRec
from replay.nn.lightning import LightningModule

NUM_UNIQUE_ITEMS = len(item_enc.classes_)

tensor_schema = TensorSchema(
    [
        TensorFeatureInfo(
            name="item_id",
            is_seq=True,
            padding_value=NUM_UNIQUE_ITEMS,
            cardinality=NUM_UNIQUE_ITEMS,
            embedding_dim=EMBEDDING_DIM,
            feature_type=FeatureType.CATEGORICAL,
            feature_hint=FeatureHint.ITEM_ID,
        )
    ]
)

transforms = make_default_sasrec_transforms(tensor_schema)

train_metadata = {
    "train": {
        "item_id": {"shape": MAX_SEQ_LEN + 1, "padding": tensor_schema["item_id"].padding_value},
    },
    "validate": {
        "item_id": {"shape": MAX_SEQ_LEN, "padding": tensor_schema["item_id"].padding_value},
        "ground_truth": {"shape": 1, "padding": -1}
    },
}

parquet_module = ParquetModule(
    train_path=TRAIN_PATH,
    validate_path=VAL_PATH,
    batch_size=BATCH_SIZE,
    metadata=train_metadata,
    transforms=transforms,
)

sasrec = SasRec.from_params(
    schema=tensor_schema,
    embedding_dim=EMBEDDING_DIM,
    max_sequence_length=MAX_SEQ_LEN,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    dropout=DROPOUT,
)
model = LightningModule(sasrec)

/tmp/ipykernel_2031/2534147900.py:41: UserWarning: The following dataset paths aren't provided: test,predict. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(


In [7]:
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger
from replay.nn.lightning.callback import ComputeMetricsCallback
import lightning as L
import torch

torch.set_float32_matmul_precision("medium")

csv_logger = CSVLogger(save_dir=f"{ROOT}/sasrec/logs/train", name="SasRec-example")

checkpoint_callback = ModelCheckpoint(
    dirpath=f"{ROOT}/sasrec/checkpoints",
    save_top_k=1,
    monitor="train_loss_epoch",
    mode="min",
    verbose=True,
)

early_stop_callback = EarlyStopping(
    monitor="train_loss_epoch",
    mode="min",
    patience=3,
    min_delta=0.001,
)

trainer = L.Trainer(
    max_epochs=50,
    callbacks=[checkpoint_callback, early_stop_callback],
    logger=csv_logger,
    enable_progress_bar=True,
    enable_model_summary=True,
    log_every_n_steps=50,
)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


# 4. Train

In [18]:
trainer.fit(model, datamodule=parquet_module)

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /content/drive/MyDrive/Projects/ml-group-project/sasrec/checkpoints exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ SasRec │  5.5 M │ train │     0 │
└───┴───────┴────────┴────────┴───────┴───────┘

Trainable params: 5.5 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 5.5 M                                                                                                
Total estimated model params size (MB): 21                                                                         
Modules in train mode: 40                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has 
`__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be 
inaccurate if each worker is not configured independently to avoid having duplicate data.

INFO: Epoch 0, global step 6280: 'train_loss_epoch' reached 6.77070 (best 6.77070), saving model to '/content/drive/MyDrive/Projects/ml-group-project/sasrec/checkpoints/epoch=0-step=6280.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 0, global step 6280: 'train_loss_epoch' reached 6.77070 (best 6.77070), saving model to '/content/drive/MyDrive/Projects/ml-group-project/sasrec/checkpoints/epoch=0-step=6280.ckpt' as top 1
INFO: Epoch 1, global step 12560: 'train_loss_epoch' reached 6.44442 (best 6.44442), saving model to '/content/drive/MyDrive/Projects/ml-group-project/sasrec/checkpoints/epoch=1-step=12560.ckpt' as top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 1, global step 12560: 'train_loss_epoch' reached 6.44442 (best 6.44442), saving model to '/content/drive/MyDrive/Projects/ml-group-project/sasrec/checkpoints/epoch=1-step=12560.ckpt' as top 1
INFO: Epoch 2, global step 18840: 'train_loss_epoch' reached 6.40525 (best 6.40525), saving model to '/content/dri

# 5. Evaluate

In [19]:
best_model_path = checkpoint_callback.best_model_path
best_epoch = int(best_model_path.split("epoch=")[1].split("-")[0])

best_model_path, best_epoch

('/content/drive/MyDrive/Projects/ml-group-project/sasrec/checkpoints/epoch=15-step=100480-v1.ckpt',
 15)

In [20]:
sasrec = SasRec.from_params(
    schema=tensor_schema,
    embedding_dim=EMBEDDING_DIM,
    max_sequence_length=MAX_SEQ_LEN,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    dropout=DROPOUT,
    excluded_features=None,
)

best_model = LightningModule.load_from_checkpoint(best_model_path, model=sasrec)
best_model.eval()

LightningModule(
  (model): SasRec(
    (body): SasRecBody(
      (embedder): SequenceEmbedding(
        (feature_embedders): ModuleDict(
          (item_id): CategoricalEmbedding(
            (emb): Embedding(84433, 64, padding_idx=84432)
          )
        )
      )
      (attn_mask_builder): DefaultAttentionMask()
      (embedding_aggregator): PositionAwareAggregator(
        (embedding_aggregator): SumAggregator()
        (pe): Embedding(50, 64)
        (dropout): Dropout(p=0.3, inplace=False)
      )
      (encoder): SasRecTransformerLayer(
        (attention_layers): ModuleList(
          (0-1): 2 x MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
          )
        )
        (attention_layernorms): ModuleList(
          (0-1): 2 x LayerNorm((64,), eps=1e-08, elementwise_affine=True)
        )
        (forward_layers): ModuleList(
          (0-1): 2 x PointWiseFeedForward(
            (conv1): Conv1d(64, 64,

In [21]:
inference_metadata = {
    "predict": {
        "user_id": {},
        "item_id": {"shape": MAX_SEQ_LEN, "padding": tensor_schema["item_id"].padding_value},
    }
}

parquet_module = ParquetModule(
    predict_path=PREDICT_PATH,
    batch_size=BATCH_SIZE,
    metadata=inference_metadata,
    transforms=transforms,
)

/tmp/ipykernel_2975/3991935412.py:8: UserWarning: The following dataset paths aren't provided: train,validate,test. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(


In [22]:
from replay.nn.lightning.callback import PandasTopItemsCallback

csv_logger = CSVLogger(save_dir="sasrec/logs/test", name="SasRec-example")

TOPK = [1, 5, 10, 20]

pandas_prediction_callback = PandasTopItemsCallback(
    top_k=max(TOPK),
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
)

trainer = L.Trainer(callbacks=[pandas_prediction_callback], logger=csv_logger, inference_mode=True)
trainer.predict(best_model, datamodule=parquet_module, return_predictions=False)

pandas_res = pandas_prediction_callback.get_result()
pandas_res

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.


,user_id,item_id,score
0,1,1247,4.930285
0,1,1197,4.784904
0,1,905,4.688539
0,1,1856,4.516138
0,1,1653,4.404428
...,...,...,...
200853,200948,12431,6.507072
200853,200948,12025,6.479527
200853,200948,13142,6.477685
200853,200948,13381,6.465157


In [23]:
from replay.metrics import MAP, OfflineMetrics, Precision, Recall
from replay.metrics.torch_metrics_builder import metrics_to_df
import pandas as pd

test_gt = pd.read_parquet(PARQUETS / "test_gt.parquet")

result_metrics = OfflineMetrics(
    [Recall(TOPK), Precision(TOPK), MAP(TOPK)],
    query_column="user_id",
    rating_column="score",
)(pandas_res, test_gt.explode("item_id"))

metrics_to_df(result_metrics)

k,1,5,10,20
MAP,0.020492,0.044037,0.052062,0.057875
Precision,0.020492,0.018252,0.015239,0.011868
Recall,0.020492,0.091260,0.152394,0.237356


In [24]:
pandas_res_renamed = pandas_res.rename(columns={'user_id': 'userId', 'item_id': 'movieId'})
pandas_res_renamed['movieId'] = item_enc.inverse_transform(pandas_res_renamed['movieId'].astype(int))
final_recommendations = pandas_res_renamed
final_recommendations.head()

,userId,movieId,score
0,1,1280,4.930285
0,1,1228,4.784904
0,1,926,4.688539
0,1,1945,4.516138
0,1,1719,4.404428


# 6. Hybrid Architecture

## Sanity check

### Load test for SasRec



In [8]:
import pickle
import pandas as pd

with open(ENCODER_PATH / "user_encoder.pkl", "rb") as f:
    user_enc = pickle.load(f)

with open(ENCODER_PATH / "item_encoder.pkl", "rb") as f:
    item_enc = pickle.load(f)

train_df = pd.read_parquet(PARQUETS / "raw_train.parquet")
test_df = pd.read_parquet(PARQUETS / "raw_test.parquet")
val_df = pd.read_parquet(PARQUETS / "raw_val.parquet")

In [11]:
import os
import pickle
import glob
from pathlib import Path
import pandas as pd
import torch
import lightning as L
from lightning.pytorch.loggers import CSVLogger

from replay.data import FeatureHint, FeatureType
from replay.data.nn.schema import TensorFeatureInfo, TensorSchema
from replay.nn.transform.template import make_default_sasrec_transforms
from replay.data.nn import ParquetModule
from replay.nn.sequential import SasRec
from replay.nn.lightning import LightningModule
from replay.nn.lightning.callback import PandasTopItemsCallback

# --- Paths & Configuration ---
ROOT = '/content/drive/MyDrive/Projects/ml-group-project'
DATA = os.path.join(ROOT, 'data')
PARQUETS = Path(os.path.join(DATA, 'parquets'))
ENCODER_PATH = PARQUETS / "encoder"
PREDICT_PATH = PARQUETS / "test.parquet"

# Find the latest checkpoint from the training phase
CHECKPOINT_PATH = '/content/drive/MyDrive/Projects/ml-group-project/sasrec/checkpoints/epoch=15-step=100480-v1.ckpt'

# Model parameters
MAX_SEQ_LEN = 50
NUM_BLOCKS = 2
NUM_HEADS = 2
DROPOUT = 0.3
BATCH_SIZE = 512 # Increased batch size for faster inference
EMBEDDING_DIM = 64
TOP_K = 20

torch.set_float32_matmul_precision("medium")

NUM_UNIQUE_ITEMS = len(item_enc.classes_)

tensor_schema = TensorSchema(
    [
        TensorFeatureInfo(
            name="item_id",
            is_seq=True,
            padding_value=NUM_UNIQUE_ITEMS,
            cardinality=NUM_UNIQUE_ITEMS,
            embedding_dim=EMBEDDING_DIM,
            feature_type=FeatureType.CATEGORICAL,
            feature_hint=FeatureHint.ITEM_ID,
        )
    ]
)
transforms = make_default_sasrec_transforms(tensor_schema)

# --- 2. Setup DataModule on existing PREDICT_PATH ---
inference_metadata = {
    "predict": {
        "user_id": {},
        "item_id": {"shape": MAX_SEQ_LEN, "padding": tensor_schema["item_id"].padding_value},
    }
}

parquet_module = ParquetModule(
    predict_path=PREDICT_PATH,
    batch_size=BATCH_SIZE,
    metadata=inference_metadata,
    transforms=transforms,
)

# --- 3. Load Model ---
sasrec = SasRec.from_params(
    schema=tensor_schema,
    embedding_dim=EMBEDDING_DIM,
    max_sequence_length=MAX_SEQ_LEN,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    dropout=DROPOUT,
)
sasrec_model = LightningModule.load_from_checkpoint(CHECKPOINT_PATH, model=sasrec)
sasrec_model.eval()

# --- 4. Run Prediction ---
pandas_prediction_callback = PandasTopItemsCallback(
    top_k=TOP_K,
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
)

trainer = L.Trainer(
    callbacks=[pandas_prediction_callback],
    logger=CSVLogger(save_dir=f"{ROOT}/sasrec/logs/inference", name="SasRec-standalone-inference"),
    inference_mode=True,
    enable_progress_bar=True
)

trainer.predict(model, datamodule=parquet_module, return_predictions=False)

# --- 5. Retrieve & Inverse Transform Results ---
pandas_res = pandas_prediction_callback.get_result()
pandas_res_renamed = pandas_res.rename(columns={'user_id': 'userId', 'item_id': 'movieId'})
pandas_res_renamed['movieId'] = item_enc.inverse_transform(pandas_res_renamed['movieId'].astype(int))

# --- 6. Query a user to predict on ---
# Let's extract recommendations for a single specific user as requested
target_user_id = pandas_res_renamed['userId'].iloc[0]
single_user_predictions = pandas_res_renamed[pandas_res_renamed['userId'] == target_user_id]

print(f"\nTop {TOP_K} ranked movie candidates for user {target_user_id} generated successfully:")
display(single_user_predictions)

/tmp/ipykernel_2031/757176643.py:64: UserWarning: The following dataset paths aren't provided: train,validate,test. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.



Top 20 ranked movie candidates for user 1 generated successfully:


,userId,movieId,score
0,1,276445,0.174717
0,1,270104,0.167891
0,1,281010,0.164294
0,1,155790,0.161051
0,1,139687,0.15677
0,1,120625,0.156325
0,1,276657,0.156075
0,1,155952,0.156072
0,1,74297,0.15503
0,1,181035,0.149747


### Load test for LightGCN

In [12]:
import torch.nn as nn

class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, num_layers, norm_adj, device):
        super(LightGCN, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers
        self.device = device
        self.register_buffer('norm_adj', norm_adj)
        self.user_embeddings = nn.Embedding(num_users, embedding_dim)
        self.item_embeddings = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.user_embeddings.weight, std=0.01)                      # Initialize user embeddings with a normal distribution (mean=0, std=0.01) for small random values
        nn.init.normal_(self.item_embeddings.weight, std=0.01)                      # Initialize item embeddings with a normal distribution (mean=0, std=0.01) for small random values

    def forward(self, users, pos_items, neg_items):
        all_embeddings = torch.cat([self.user_embeddings.weight, self.item_embeddings.weight], dim=0)
        ego_embeddings = all_embeddings
        for _ in range(self.num_layers):                                            # Loop over the specified number of graph convolution layers
            all_embeddings = torch.spmm(self.norm_adj, all_embeddings)              # Perform sparse matrix multiplication with normalized adjacency matrix to propagate embeddings
            ego_embeddings += all_embeddings                                        # Add the propagated embeddings to the running sum (LightGCN aggregates all layers)
        final_embeddings = ego_embeddings / (self.num_layers + 1)                   # Average the embeddings across all layers (including ego layer) for smoothness
        user_emb = final_embeddings[users]
        pos_item_emb = final_embeddings[self.num_users + pos_items]                 # because we sampled pos_items from items (0 to 1681) we have to offset by the number of users
        neg_item_emb = final_embeddings[self.num_users + neg_items]                 # because we sampled neg_items from items (0 to 1681) we have to offset by the number of users
        pos_scores = (user_emb * pos_item_emb).sum(dim=-1)                          # predicted scores of positive samples
        neg_scores = (user_emb * neg_item_emb).sum(dim=-1)                          # predicted scores of negative samples

        return pos_scores, neg_scores

    def get_embeddings(self):
        all_embeddings = torch.cat([self.user_embeddings.weight, self.item_embeddings.weight], dim=0)
        ego_embeddings = all_embeddings
        for _ in range(self.num_layers):
            all_embeddings = torch.spmm(self.norm_adj, all_embeddings)
            ego_embeddings += all_embeddings
        final_embeddings = ego_embeddings / (self.num_layers + 1)
        user_emb = final_embeddings[:self.num_users]
        item_emb = final_embeddings[self.num_users:]
        return user_emb, item_emb

In [13]:
LIGHTGCN_SAVE_PATH = os.path.join(ROOT, 'lightgcn_weights.pth')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_df["user_idx"] = user_enc.transform(train_df["userId"])
train_df["item_idx"] = item_enc.transform(train_df["movieId"])
test_df["user_idx"] = user_enc.transform(test_df["userId"])
test_df["item_idx"] = item_enc.transform(test_df["movieId"])
val_df["user_idx"] = user_enc.transform(val_df["userId"])
val_df["item_idx"] = item_enc.transform(val_df["movieId"])

train_df["user_idx"] += 1
test_df["user_idx"] += 1
val_df["user_idx"] += 1

num_users = len(user_enc.classes_)
num_items = len(item_enc.classes_)

train_interactions = torch.tensor(
    train_df[["user_idx", "item_idx"]].values,
    dtype=torch.long
)
test_interactions = torch.tensor(
    test_df[["user_idx", "item_idx"]].values,
    dtype=torch.long
)

rows = torch.cat([train_interactions[:, 0], train_interactions[:, 1] + num_users], dim=0)
cols = torch.cat([train_interactions[:, 1] + num_users, train_interactions[:, 0]], dim=0)
indices = torch.stack([rows, cols], dim=0).to(device)
values = torch.ones(indices.shape[1], device=device)
adj = torch.sparse_coo_tensor(indices, values, size=(num_users + num_items, num_users + num_items), device=device)
degrees = torch.sparse.sum(adj, dim=1).to_dense()
norm_values = 1.0 / (torch.sqrt(degrees[rows]) * torch.sqrt(degrees[cols])).to(device)
norm_adj = torch.sparse_coo_tensor(indices, norm_values, size=(num_users + num_items, num_users + num_items), device=device)

# Hyperparameters
embedding_dim = 64
num_layers = 3
learning_rate = 1e-3
num_epochs = 100
lambda_reg = 1e-6  # Regularization weight

lightgcn_model = LightGCN(num_users, num_items, embedding_dim, num_layers, norm_adj, device)
lightgcn_model.load_state_dict(torch.load(LIGHTGCN_SAVE_PATH))
lightgcn_model.to(device)
lightgcn_model.eval()

# Extract embeddings
with torch.no_grad():
    user_emb, item_emb = lightgcn_model.get_embeddings()

# print("User embedding shape:", user_emb.shape)
# print("Item embedding shape:", item_emb.shape)

# Confirm shapes match encoder sizes
expected_num_users = len(user_enc.classes_)
expected_num_items = len(item_enc.classes_)

assert user_emb.shape[0] == expected_num_users, (
    f"user_emb has {user_emb.shape[0]} rows, expected {expected_num_users}"
)
assert item_emb.shape[0] == expected_num_items, (
    f"item_emb has {item_emb.shape[0]} rows, expected {expected_num_items}"
)
assert user_emb.shape[1] == embedding_dim
assert item_emb.shape[1] == embedding_dim

print("Embedding shapes match encoder sizes.")

/tmp/ipykernel_2031/2022160482.py:31: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  adj = torch.sparse_coo_tensor(indices, values, size=(num_users + num_items, num_users + num_items), device=device)


Embedding shapes match encoder sizes.


## Freeze SasRec as candidate generator

In [19]:
import numpy as np
import torch

sasrec_model = sasrec_model.to(device)
sasrec_model.eval()

# Freeze parameters
for p in sasrec_model.parameters():
    p.requires_grad = False

# Define user_id for testing
user_id = 1
top_k = 100

# Get user's history from training data only
user_hist = (
    train_df[train_df["userId"] == user_id]
    .sort_values("timestamp")["movieId"]
    .tolist()
)

if len(user_hist) == 0:
    candidates = []
else:
    # Encode item IDs into SASRec indices
    hist_idx = item_enc.transform(user_hist)

    # Pad on the left so the most recent items are at the end
    pad_idx = len(item_enc.classes_)
    seq = np.full(MAX_SEQ_LEN, pad_idx, dtype=np.int64)
    seq[-min(len(hist_idx), MAX_SEQ_LEN):] = hist_idx[-MAX_SEQ_LEN:]

    seq = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        padding_mask = (seq == pad_idx)
        batch = {
            "feature_tensors": {"item_id": seq},
            "padding_mask": padding_mask
        }

        output = sasrec_model(batch)

        if isinstance(output, tuple):
            scores = output[0]
        elif hasattr(output, 'pred_scores'):
            scores = output.pred_scores
        elif hasattr(output, 'logits'):
            scores = output.logits
        elif isinstance(output, dict) and 'logits' in output:
            scores = output['logits']
        elif isinstance(output, dict) and 'scores' in output:
            scores = output['scores']
        else:
            scores = output

        # scores shape should be [batch, num_items]
        scores = scores.squeeze(0)

        # Remove already-seen items from recommendations
        seen = set(hist_idx.tolist())
        if seen:
            seen_idx = torch.tensor(list(seen), device=device)
            scores[seen_idx] = float("-inf")

        top_scores, top_items = torch.topk(scores, k=top_k)
        top_items = top_items.cpu().numpy()
        top_scores = top_scores.cpu().numpy()

    # Map back to original movieIds
    movie_ids = item_enc.inverse_transform(top_items)

    candidates = list(zip(movie_ids.tolist(), top_scores.tolist()))
    print(f"Top {top_k} candidates for user {user_id}:")
    print(candidates)


Top 100 candidates for user 1:
[(318, 0.8010542392730713), (1193, 0.7091450691223145), (1230, 0.6611397862434387), (858, 0.5186887383460999), (904, 0.4830579459667206), (903, 0.2789079248905182), (913, 0.2720303237438202), (2959, 0.15656940639019012), (750, -0.0030268209520727396), (1212, -0.07920422405004501), (1252, -0.10169529914855957), (1198, -0.1123868003487587), (922, -0.16832931339740753), (2858, -0.20765532553195953), (1, -0.22800429165363312), (296, -0.3539038896560669), (924, -0.4495947062969208), (950, -0.47974979877471924), (920, -0.4896252155303955), (910, -0.4922337830066681), (50, -0.5360334515571594), (3089, -0.6410232782363892), (2731, -0.6662372350692749), (898, -0.6836028099060059), (1104, -0.6919119954109192), (1219, -0.7075797319412231), (1207, -0.7117273807525635), (951, -0.7708384394645691), (1617, -0.787526547908783), (953, -0.8113201856613159), (3095, -0.8598589301109314), (969, -0.8636606335639954), (899, -0.8765135407447815), (150, -0.9054539799690247), (201

In [20]:
alpha = 0.7

# candidates is a list of (movie_id, sasrec_score)
candidate_movie_ids = [c[0] for c in candidates]
sasrec_scores = np.array([c[1] for c in candidates])

# Get user and item indices for LightGCN
u_idx = user_enc.transform([user_id])[0]
i_idxs = item_enc.transform(candidate_movie_ids)

# Retrieve embeddings from LightGCN
# user_emb and item_emb were extracted in the earlier LightGCN cell
u_emb = user_emb[u_idx] # shape: (embedding_dim,)
i_embs = item_emb[i_idxs] # shape: (num_candidates, embedding_dim)

# Compute LightGCN scores via dot product
lightgcn_scores = (u_emb.unsqueeze(0) * i_embs).sum(dim=-1).cpu().numpy()

# Combine scores using the fusion formula
final_scores = alpha * sasrec_scores + (1 - alpha) * lightgcn_scores

# Re-rank candidates
fused_candidates = list(zip(candidate_movie_ids, final_scores))
fused_candidates.sort(key=lambda x: x[1], reverse=True)

print(f"Top 20 candidates after fusion (alpha={alpha}) for user {user_id}:")
for i, (m_id, score) in enumerate(fused_candidates[:20]):
    print(f"{i+1}. Movie ID: {m_id}, Fused Score: {score:.4f}")


Top 20 candidates after fusion (alpha=0.7) for user 1:
1. Movie ID: 318, Fused Score: 0.5607
2. Movie ID: 1193, Fused Score: 0.4964
3. Movie ID: 1230, Fused Score: 0.4628
4. Movie ID: 858, Fused Score: 0.3631
5. Movie ID: 904, Fused Score: 0.3381
6. Movie ID: 903, Fused Score: 0.1952
7. Movie ID: 913, Fused Score: 0.1904
8. Movie ID: 2959, Fused Score: 0.1096
9. Movie ID: 750, Fused Score: -0.0021
10. Movie ID: 1212, Fused Score: -0.0554
11. Movie ID: 1252, Fused Score: -0.0712
12. Movie ID: 1198, Fused Score: -0.0787
13. Movie ID: 922, Fused Score: -0.1178
14. Movie ID: 2858, Fused Score: -0.1454
15. Movie ID: 1, Fused Score: -0.1596
16. Movie ID: 296, Fused Score: -0.2477
17. Movie ID: 924, Fused Score: -0.3147
18. Movie ID: 950, Fused Score: -0.3358
19. Movie ID: 920, Fused Score: -0.3427
20. Movie ID: 910, Fused Score: -0.3446


In [24]:
import numpy as np
import torch
from tqdm.auto import tqdm

# alphas to test
alphas = [0.3, 0.5, 0.7, 0.9]
val_results = {alpha: {'recall': [], 'ndcg': []} for alpha in alphas}

# Sample validation users to keep tuning fast
np.random.seed(42)
unique_val_users = val_df['userId'].unique()
val_sampled_users = np.random.choice(unique_val_users, size=500, replace=False)

TOP_K = 10
SASREC_CAND_K = 100

sasrec_model.eval()

for user_id in tqdm(val_sampled_users, desc="Tuning alpha on validation"):
    # Ground truth for validation is in val_df
    gt_items = val_df[val_df['userId'] == user_id]['movieId'].tolist()
    if not gt_items:
        continue

    # User history from train ONLY (to predict validation)
    user_hist = train_df[train_df['userId'] == user_id].sort_values('timestamp')['movieId'].tolist()
    if not user_hist:
        continue

    # 1. SASRec Prediction
    hist_idx = item_enc.transform(user_hist)
    pad_idx = len(item_enc.classes_)
    seq = np.full(MAX_SEQ_LEN, pad_idx, dtype=np.int64)
    seq[-min(len(hist_idx), MAX_SEQ_LEN):] = hist_idx[-MAX_SEQ_LEN:]
    seq_tensor = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        padding_mask = (seq_tensor == pad_idx)
        batch = {
            "feature_tensors": {"item_id": seq_tensor},
            "padding_mask": padding_mask
        }
        output = sasrec_model(batch)

        if isinstance(output, tuple):
            scores = output[0]
        elif hasattr(output, 'pred_scores'):
            scores = output.pred_scores
        elif hasattr(output, 'logits'):
            scores = output.logits
        elif isinstance(output, dict) and 'logits' in output:
            scores = output['logits']
        elif isinstance(output, dict) and 'scores' in output:
            scores = output['scores']
        else:
            scores = output

        scores = scores.squeeze(0)

        # Mask seen items in train
        seen = set(hist_idx.tolist())
        if seen:
            seen_idx = torch.tensor(list(seen), device=device)
            scores[seen_idx] = float("-inf")

        top_scores, top_items = torch.topk(scores, k=SASREC_CAND_K)
        top_items_np = top_items.cpu().numpy()
        sasrec_scores_np = top_scores.cpu().numpy()

    # 2. LightGCN Prediction for the SASRec candidates
    try:
        u_idx = user_enc.transform([user_id])[0]
        u_emb = user_emb[u_idx]

        cand_items_idx = top_items_np
        cand_i_embs = item_emb[cand_items_idx]

        lgcn_cand_scores = (u_emb.unsqueeze(0) * cand_i_embs).sum(dim=-1).cpu().numpy()
        sasrec_cand_scores = sasrec_scores_np

        # Evaluate each alpha
        for alpha in alphas:
            hybrid_scores = alpha * sasrec_cand_scores + (1 - alpha) * lgcn_cand_scores

            hybrid_cands = list(zip(cand_items_idx, hybrid_scores))
            hybrid_cands.sort(key=lambda x: x[1], reverse=True)

            hybrid_rec_idxs = [c[0] for c in hybrid_cands[:TOP_K]]
            hybrid_rec_ids = item_enc.inverse_transform(hybrid_rec_idxs)

            r, n, _ = get_metrics(hybrid_rec_ids, gt_items, TOP_K)
            val_results[alpha]['recall'].append(r)
            val_results[alpha]['ndcg'].append(n)

    except ValueError:
        # User not in LightGCN encoder, fallback to pure SASRec
        sasrec_rec_ids = item_enc.inverse_transform(top_items_np[:TOP_K])
        for alpha in alphas:
            r, n, _ = get_metrics(sasrec_rec_ids, gt_items, TOP_K)
            val_results[alpha]['recall'].append(r)
            val_results[alpha]['ndcg'].append(n)

print("\n=== Alpha Tuning Results on Validation ===")
best_alpha = None
best_ndcg = -1

for alpha in alphas:
    avg_recall = np.mean(val_results[alpha]['recall'])
    avg_ndcg = np.mean(val_results[alpha]['ndcg'])
    print(f"Alpha {alpha:.1f} -> Recall@10: {avg_recall:.4f} | NDCG@10: {avg_ndcg:.4f}")

    if avg_ndcg > best_ndcg:
        best_ndcg = avg_ndcg
        best_alpha = alpha

print(f"\nBest Alpha based on NDCG@10: {best_alpha:.1f}")


Tuning alpha on validation:   0%|          | 0/500 [00:00<?, ?it/s]


=== Alpha Tuning Results on Validation ===
Alpha 0.3 -> Recall@10: 0.0360 | NDCG@10: 0.0191
Alpha 0.5 -> Recall@10: 0.0440 | NDCG@10: 0.0231
Alpha 0.7 -> Recall@10: 0.0560 | NDCG@10: 0.0270
Alpha 0.9 -> Recall@10: 0.0600 | NDCG@10: 0.0297

Best Alpha based on NDCG@10: 0.9


In [25]:
import numpy as np
import torch

def recommend_hybrid(user_id, user_history, top_k=10, sasrec_cand_k=100, alpha=0.7):
    """
    Generates hybrid recommendations by first retrieving candidates from SASRec,
    then re-scoring them using LightGCN embeddings.
    """
    if len(user_history) == 0:
        return []

    # 1. Encode the user history
    hist_idx = item_enc.transform(user_history)

    pad_idx = len(item_enc.classes_)
    seq = np.full(MAX_SEQ_LEN, pad_idx, dtype=np.int64)
    seq[-min(len(hist_idx), MAX_SEQ_LEN):] = hist_idx[-MAX_SEQ_LEN:]
    seq = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)

    # 2. Get SASRec candidates
    with torch.no_grad():
        padding_mask = (seq == pad_idx)
        batch = {
            "feature_tensors": {"item_id": seq},
            "padding_mask": padding_mask
        }

        output = sasrec_model(batch)

        if isinstance(output, tuple):
            scores = output[0]
        elif hasattr(output, 'pred_scores'):
            scores = output.pred_scores
        elif hasattr(output, 'logits'):
            scores = output.logits
        elif isinstance(output, dict) and 'logits' in output:
            scores = output['logits']
        elif isinstance(output, dict) and 'scores' in output:
            scores = output['scores']
        else:
            scores = output

        scores = scores.squeeze(0)

        # Mask already-seen items
        seen = set(hist_idx.tolist())
        if seen:
            seen_idx = torch.tensor(list(seen), device=device)
            scores[seen_idx] = float("-inf")

        top_scores, top_items = torch.topk(scores, k=sasrec_cand_k)
        top_items = top_items.cpu().numpy()
        sasrec_scores = top_scores.cpu().numpy()

    candidate_movie_ids = item_enc.inverse_transform(top_items)

    # 3. Score those candidates with LightGCN
    try:
        u_idx = user_enc.transform([user_id])[0]
    except ValueError:
        # Fallback if user is completely unknown to LightGCN (cold start)
        fused_candidates = list(zip(candidate_movie_ids, sasrec_scores))
        return fused_candidates[:top_k]

    i_idxs = item_enc.transform(candidate_movie_ids)

    u_emb = user_emb[u_idx] # shape: (embedding_dim,)
    i_embs = item_emb[i_idxs] # shape: (sasrec_cand_k, embedding_dim)

    lightgcn_scores = (u_emb.unsqueeze(0) * i_embs).sum(dim=-1).cpu().numpy()

    # 4. Return final ranked movies
    final_scores = alpha * sasrec_scores + (1 - alpha) * lightgcn_scores

    fused_candidates = list(zip(candidate_movie_ids, final_scores))
    fused_candidates.sort(key=lambda x: x[1], reverse=True)

    return fused_candidates[:top_k]

# --- Test the function ---
test_user = 1
test_history = train_df[train_df["userId"] == test_user].sort_values("timestamp")["movieId"].tolist()
hybrid_recs = recommend_hybrid(test_user, test_history, top_k=10, alpha=0.9)

print(f"Hybrid Top 10 Recommendations for User {test_user}:")
for i, (m_id, score) in enumerate(hybrid_recs):
    print(f"{i+1}. Movie ID: {m_id}, Fused Score: {score:.4f}")


Hybrid Top 10 Recommendations for User 1:
1. Movie ID: 318, Fused Score: 0.7209
2. Movie ID: 1193, Fused Score: 0.6382
3. Movie ID: 1230, Fused Score: 0.5950
4. Movie ID: 858, Fused Score: 0.4668
5. Movie ID: 904, Fused Score: 0.4348
6. Movie ID: 903, Fused Score: 0.2510
7. Movie ID: 913, Fused Score: 0.2448
8. Movie ID: 2959, Fused Score: 0.1409
9. Movie ID: 750, Fused Score: -0.0027
10. Movie ID: 1212, Fused Score: -0.0713


In [26]:
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import math

def get_metrics(ranked_list, ground_truth, k=10):
    hits = 0
    dcg = 0
    idcg = 0

    # Calculate IDCG@k
    for i in range(min(len(ground_truth), k)):
        idcg += 1 / math.log2(i + 2)

    if idcg == 0:
        return 0.0, 0.0, 0.0

    for i, item in enumerate(ranked_list[:k]):
        if item in ground_truth:
            hits += 1
            dcg += 1 / math.log2(i + 2)

    recall = hits / len(ground_truth)
    ndcg = dcg / idcg
    hr = 1.0 if hits > 0 else 0.0

    return recall, ndcg, hr

# Prepare test data
# Let's take a random sample of 500 users to keep evaluation time reasonable
np.random.seed(42)
unique_test_users = test_df['userId'].unique()
sampled_users = np.random.choice(unique_test_users, size=500, replace=False)

results = {
    'SASRec': {'recall': [], 'ndcg': [], 'hr': []},
    'LightGCN': {'recall': [], 'ndcg': [], 'hr': []},
    'Hybrid': {'recall': [], 'ndcg': [], 'hr': []}
}

TOP_K = 10
ALPHA = 0.9
SASREC_CAND_K = 100

sasrec_model.eval()

for user_id in tqdm(sampled_users, desc="Evaluating users"):
    # Ground truth
    gt_items = test_df[test_df['userId'] == user_id]['movieId'].tolist()
    if not gt_items:
        continue

    # User history from train & val
    user_hist = train_df[train_df['userId'] == user_id].sort_values('timestamp')['movieId'].tolist()
    val_hist = val_df[val_df['userId'] == user_id].sort_values('timestamp')['movieId'].tolist()
    full_hist = user_hist + val_hist

    if not full_hist:
        continue

    # 1. SASRec Prediction
    hist_idx = item_enc.transform(full_hist)
    pad_idx = len(item_enc.classes_)
    seq = np.full(MAX_SEQ_LEN, pad_idx, dtype=np.int64)
    seq[-min(len(hist_idx), MAX_SEQ_LEN):] = hist_idx[-MAX_SEQ_LEN:]
    seq_tensor = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        padding_mask = (seq_tensor == pad_idx)
        batch = {
            "feature_tensors": {"item_id": seq_tensor},
            "padding_mask": padding_mask
        }
        output = sasrec_model(batch)

        if isinstance(output, tuple):
            scores = output[0]
        elif hasattr(output, 'pred_scores'):
            scores = output.pred_scores
        elif hasattr(output, 'logits'):
            scores = output.logits
        elif isinstance(output, dict) and 'logits' in output:
            scores = output['logits']
        elif isinstance(output, dict) and 'scores' in output:
            scores = output['scores']
        else:
            scores = output

        scores = scores.squeeze(0)

        seen = set(hist_idx.tolist())
        if seen:
            seen_idx = torch.tensor(list(seen), device=device)
            scores[seen_idx] = float("-inf")

        top_scores, top_items = torch.topk(scores, k=max(TOP_K, SASREC_CAND_K))
        top_items_np = top_items.cpu().numpy()
        sasrec_scores_np = top_scores.cpu().numpy()

    sasrec_rec_ids = item_enc.inverse_transform(top_items_np[:TOP_K])

    # 2. LightGCN Prediction
    try:
        u_idx = user_enc.transform([user_id])[0]
        u_emb = user_emb[u_idx]

        # For pure LightGCN, we score all items
        all_items_idx = torch.arange(num_items, device=device)
        all_i_embs = item_emb

        lgcn_all_scores = (u_emb.unsqueeze(0) * all_i_embs).sum(dim=-1)

        # Mask seen
        if seen:
            lgcn_all_scores[seen_idx] = float("-inf")

        lgcn_top_scores, lgcn_top_items = torch.topk(lgcn_all_scores, k=TOP_K)
        lgcn_rec_ids = item_enc.inverse_transform(lgcn_top_items.cpu().numpy())

        # 3. Hybrid Prediction
        # Score the top SASREC_CAND_K candidates from SASRec
        cand_items_idx = top_items_np[:SASREC_CAND_K]
        cand_i_embs = item_emb[cand_items_idx]

        lgcn_cand_scores = (u_emb.unsqueeze(0) * cand_i_embs).sum(dim=-1).cpu().numpy()
        sasrec_cand_scores = sasrec_scores_np[:SASREC_CAND_K]

        hybrid_scores = ALPHA * sasrec_cand_scores + (1 - ALPHA) * lgcn_cand_scores

        hybrid_cands = list(zip(cand_items_idx, hybrid_scores))
        hybrid_cands.sort(key=lambda x: x[1], reverse=True)

        hybrid_rec_idxs = [c[0] for c in hybrid_cands[:TOP_K]]
        hybrid_rec_ids = item_enc.inverse_transform(hybrid_rec_idxs)

    except ValueError:
        # User not in LightGCN encoder
        lgcn_rec_ids = []
        hybrid_rec_ids = sasrec_rec_ids

    # Calculate metrics
    s_r, s_n, s_h = get_metrics(sasrec_rec_ids, gt_items, TOP_K)
    results['SASRec']['recall'].append(s_r)
    results['SASRec']['ndcg'].append(s_n)
    results['SASRec']['hr'].append(s_h)

    l_r, l_n, l_h = get_metrics(lgcn_rec_ids, gt_items, TOP_K)
    results['LightGCN']['recall'].append(l_r)
    results['LightGCN']['ndcg'].append(l_n)
    results['LightGCN']['hr'].append(l_h)

    h_r, h_n, h_h = get_metrics(hybrid_rec_ids, gt_items, TOP_K)
    results['Hybrid']['recall'].append(h_r)
    results['Hybrid']['ndcg'].append(h_n)
    results['Hybrid']['hr'].append(h_h)

# Aggregate and Print Results
print("\n=== Evaluation Results (@10) ===")
for model_name, metrics in results.items():
    avg_recall = np.mean(metrics['recall'])
    avg_ndcg = np.mean(metrics['ndcg'])
    avg_hr = np.mean(metrics['hr'])

    print(f"{model_name}:")
    print(f"  Recall@10: {avg_recall:.4f}")
    print(f"  NDCG@10:   {avg_ndcg:.4f}")
    print(f"  HR@10:     {avg_hr:.4f}")


Evaluating users:   0%|          | 0/500 [00:00<?, ?it/s]


=== Evaluation Results (@10) ===
SASRec:
  Recall@10: 0.0480
  NDCG@10:   0.0242
  HR@10:     0.0480
LightGCN:
  Recall@10: 0.0300
  NDCG@10:   0.0147
  HR@10:     0.0300
Hybrid:
  Recall@10: 0.0460
  NDCG@10:   0.0229
  HR@10:     0.0460


In [27]:
import json
import numpy as np
import torch
import os

# Randomly select 3 users for the demo
np.random.seed(42)
demo_users = np.random.choice(sampled_users, size=3, replace=False)
demo_results = {}

TOP_K = 10
ALPHA = 0.9
SASREC_CAND_K = 100

sasrec_model.eval()

for uid in demo_users:
    user_id = int(uid)

    # Get User history from train & val
    user_hist = train_df[train_df['userId'] == user_id].sort_values('timestamp')['movieId'].tolist()
    val_hist = val_df[val_df['userId'] == user_id].sort_values('timestamp')['movieId'].tolist()
    full_hist = user_hist + val_hist

    if not full_hist:
        continue

    # 1. SASRec Prediction
    hist_idx = item_enc.transform(full_hist)
    pad_idx = len(item_enc.classes_)
    seq = np.full(MAX_SEQ_LEN, pad_idx, dtype=np.int64)
    seq[-min(len(hist_idx), MAX_SEQ_LEN):] = hist_idx[-MAX_SEQ_LEN:]
    seq_tensor = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        padding_mask = (seq_tensor == pad_idx)
        batch = {
            "feature_tensors": {"item_id": seq_tensor},
            "padding_mask": padding_mask
        }
        output = sasrec_model(batch)

        if isinstance(output, tuple):
            scores = output[0]
        elif hasattr(output, 'pred_scores'):
            scores = output.pred_scores
        elif hasattr(output, 'logits'):
            scores = output.logits
        elif isinstance(output, dict) and 'logits' in output:
            scores = output['logits']
        elif isinstance(output, dict) and 'scores' in output:
            scores = output['scores']
        else:
            scores = output

        scores = scores.squeeze(0)

        seen = set(hist_idx.tolist())
        if seen:
            seen_idx = torch.tensor(list(seen), device=device)
            scores[seen_idx] = float("-inf")

        top_scores, top_items = torch.topk(scores, k=max(TOP_K, SASREC_CAND_K))
        top_items_np = top_items.cpu().numpy()
        sasrec_scores_np = top_scores.cpu().numpy()

    sasrec_rec_ids = item_enc.inverse_transform(top_items_np[:TOP_K]).tolist()

    # 2. LightGCN Prediction
    try:
        u_idx = user_enc.transform([user_id])[0]
        u_emb = user_emb[u_idx]

        # For pure LightGCN, we score all items
        all_items_idx = torch.arange(num_items, device=device)
        all_i_embs = item_emb

        lgcn_all_scores = (u_emb.unsqueeze(0) * all_i_embs).sum(dim=-1)

        # Mask seen
        if seen:
            lgcn_all_scores[seen_idx] = float("-inf")

        lgcn_top_scores, lgcn_top_items = torch.topk(lgcn_all_scores, k=TOP_K)
        lgcn_rec_ids = item_enc.inverse_transform(lgcn_top_items.cpu().numpy()).tolist()

        # 3. Hybrid Prediction
        cand_items_idx = top_items_np[:SASREC_CAND_K]
        cand_i_embs = item_emb[cand_items_idx]

        lgcn_cand_scores = (u_emb.unsqueeze(0) * cand_i_embs).sum(dim=-1).cpu().numpy()
        sasrec_cand_scores = sasrec_scores_np[:SASREC_CAND_K]

        hybrid_scores = ALPHA * sasrec_cand_scores + (1 - ALPHA) * lgcn_cand_scores

        hybrid_cands = list(zip(cand_items_idx, hybrid_scores))
        hybrid_cands.sort(key=lambda x: x[1], reverse=True)

        hybrid_rec_idxs = [c[0] for c in hybrid_cands[:TOP_K]]
        hybrid_rec_ids = item_enc.inverse_transform(hybrid_rec_idxs).tolist()

    except ValueError:
        lgcn_rec_ids = []
        hybrid_rec_ids = sasrec_rec_ids

    # Store the results
    demo_results[user_id] = {
        "history_last_20": full_hist[-20:], # Only keeping the last 20 for a concise report
        "sasrec_top10": sasrec_rec_ids,
        "lightgcn_top10": lgcn_rec_ids,
        "hybrid_top10": hybrid_rec_ids
    }

# Print nicely formatted JSON
print(json.dumps(demo_results, indent=2))

# Save to drive
report_path = os.path.join(ROOT, "demo_results.json")
with open(report_path, "w") as f:
    json.dump(demo_results, f, indent=2)
print(f"\nSaved demo report to {report_path}")

{
  "51128": {
    "history_last_20": [
      89898,
      92259,
      318,
      164179,
      3147,
      79132,
      2571,
      116797,
      1682,
      106100,
      99114,
      106782,
      7254,
      7147,
      55269,
      45720,
      2959,
      49530,
      4874
    ],
    "sasrec_top10": [
      2501,
      524,
      1378,
      531,
      135,
      107,
      1831,
      3555,
      2094,
      830
    ],
    "lightgcn_top10": [
      356,
      296,
      593,
      260,
      527,
      480,
      4993,
      110,
      1196,
      1
    ],
    "hybrid_top10": [
      2501,
      524,
      1378,
      531,
      135,
      107,
      1831,
      3555,
      2094,
      830
    ]
  },
  "167566": {
    "history_last_20": [
      1198,
      1197,
      50,
      81847,
      134853,
      293,
      1210,
      858,
      1291,
      4993,
      1214,
      1136,
      1172,
      92259,
      923,
      111,
      1233,
      1193,
      912,
      1237
    ],
